# Wind Load Calc — Sign on a Chain-Link Fence on a Traffic Barrier

An invented but typical detail: an **8-ft chain-link protective fence** mounted on
top of a concrete traffic barrier / retaining wall, with a **3 ft × 5 ft solid
metal sign** clamped to one of the fence posts.  We chase the wind force through
the whole load path and report a demand/capacity ratio at every link:

\# //TODO - ODOT's bridge fence height, think it's only 6'/72"

1. **Wind load** — AASHTO *LRFD Specifications for Structural Supports for
   Highway Signs, Luminaires, and Traffic Signals* (LRFDLTS-1), Article 3.8
2. **Sign-to-fence connection** — U-bolt clamps (LTS 5.15 → AISC bolt shear)
3. **Fence post** — cantilever flexure (LTS Section 5, AISC pipe section)
4. **Fabric-to-post connection** — tension bands
5. **Anchor bolts** — base plate couple → `civilpy.structural.concrete.AnchorBolts`
   (LTS 5.16.3 sends anchorage design to AISC Design Guide 1 + ACI 318 Ch. 17)

Section/table references below are to LRFDLTS-1, 1st Ed. (2015, with interims
through 2020) unless noted.  Load combination: **Extreme I** — 1.0·W with the
700-yr MRI wind map, 1.1/0.9·DC (Table 3.4-1, Table 3.8-1).

## Geometry and site

| item | value |
|---|---|
| fence fabric height above barrier, $H_f$ | 8 ft (11-ga, 1-in mesh) |
| post spacing | 8 ft o.c. |
| line posts | Pipe 2-1/2 Sch 40 (2.875 in OD) |
| sign | 3 ft tall × 5 ft wide solid panel, top at top of fence, centered on a post |
| barrier | 12-in wide top, f′c = 4,000 psi |
| site | Ohio — V(700-yr MRI) = **115 mph**, V(10-yr, Fig. 3.8-4) = 76 mph |

The wind's risk category is *Typical* (a failed post or sign could fall into the
travelway), so Table 3.8-1 gives MRI = 700 yrs for the strength (Extreme I) check.
Note that the "roadside sign supports: use 10-yr MRI" relief in Table 3.8-1 does
**not** apply — this is an attachment over traffic, not a breakaway roadside sign.

\#//TODO - Wire gauge was wrong, it's 11-gauge 1-in mesh. Updated here, ensure it's fixed throughout notebook

In [1]:
import math
from civilpy.general import units

# --- site / spec inputs -------------------------------------------------
V   = 115.0        # mph, 3-s gust, 700-yr MRI (Fig. 3.8-1b, Ohio)
V10 = 76.0         # mph, 10-yr MRI service wind (Fig. 3.8-4)

# --- geometry -----------------------------------------------------------
H_fabric   = 8.0          # ft of fabric above the barrier / base plate
s_post     = 10.0         # ft, post spacing
sign_w, sign_h = 5.0, 3.0 # ft (Lsign = 5, Wsign = 3)
A_sign     = sign_w * sign_h                    # 15 ft^2
z_sign_cg  = sign_h / 2.0            # sign centroid above base, ft
solidity   = 0.20         # projected solid fraction of 9-ga 2-in mesh fabric
                          # (CLFMI Wind Load Guide; LTS Table 3.8.7-1 note a
                          #  permits peer-reviewed values for untabulated shapes)
print(f"sign area = {A_sign:.0f} ft^2, centroid {z_sign_cg:.1f} ft above the base plate")

sign area = 15 ft^2, centroid 1.5 ft above the base plate


## Step 1 — Design wind pressure (LTS Article 3.8)

$$P_z = 0.00256\,K_z\,K_d\,G\,V^2\,C_d \quad \text{(psf, Eq. 3.8.1-1)}$$

- $K_z$ — height/exposure factor, Eq. 3.8.4-1 with $\alpha = 9.5$, $z_g = 900$ ft,
  and $z \ge 16$ ft (Exposure C is mandatory in LTS)
- $K_d$ — directionality, Table 3.8.5-1 (round pole → 0.95)
- $G$ — gust effect factor, min 1.14 (Art. 3.8.6)
- $C_d$ — Table 3.8.7-1: sign panel interpolated on $L_{sign}/W_{sign}$;
  cylindrical member by the $C_v V d$ regime ($C_v = 0.8$ at the extreme limit
  state); chain-link fabric is not tabulated, so we use net (solid) area with
  $C_d = 1.2$ per the CLFMI guide under note *a*

In [2]:
def Kz(z_ft):
    """LTS Eq. 3.8.4-1 (Exposure C: alpha = 9.5, zg = 900 ft, z >= 16 ft)."""
    z = max(z_ft, 16.0)
    return 2.0 * (z / 900.0) ** (2.0 / 9.5)

Kd = 0.95      # Table 3.8.5-1, round pole support
G  = 1.14      # Art. 3.8.6 minimum

# Cd — sign panel, Table 3.8.7-1 (interpolate on Lsign/Wsign)
ratios, cds = [1.0, 2.0, 5.0, 10.0, 15.0], [1.12, 1.19, 1.20, 1.23, 1.30]
import numpy as np
Cd_sign = float(np.interp(sign_w / sign_h, ratios, cds))

# Cd — post (single cylindrical member): regime set by Cv*V*d, Cv = 0.8 extreme
d_post_ft = 2.875 / 12.0
CvVd = 0.8 * V * d_post_ft
Cd_post = 1.10 if CvVd <= 39 else (129.0 / CvVd**1.3 if CvVd < 78 else 0.45)

Cd_fabric = 1.2   # on net (solid) area — CLFMI / note a

z_top = H_fabric + 4.0          # fence top ~4 ft barrier + 8 ft fabric above grade
kz = Kz(z_top)
q = 0.00256 * kz * Kd * G * V**2     # psf per unit Cd  (Eq. 3.8.1-1)

Pz_sign, Pz_fabric, Pz_post = q * Cd_sign, q * Cd_fabric, q * Cd_post
print(f"Kz(z={z_top:.0f} ft) = {kz:.3f}   q = 0.00256*Kz*Kd*G*V^2 = {q:.1f} psf per Cd")
print(f"Cd: sign {Cd_sign:.3f} (L/W={sign_w/sign_h:.2f}), fabric {Cd_fabric}, post {Cd_post:.2f} (CvVd={CvVd:.0f} mph-ft)")
print(f"Pz: sign {Pz_sign:.1f} psf | fabric (net area) {Pz_fabric:.1f} psf | post {Pz_post:.1f} psf")

Kz(z=12 ft) = 0.856   q = 0.00256*Kz*Kd*G*V^2 = 31.4 psf per Cd
Cd: sign 1.167 (L/W=1.67), fabric 1.2, post 1.10 (CvVd=22 mph-ft)
Pz: sign 36.6 psf | fabric (net area) 37.7 psf | post 34.5 psf


In [3]:
# Forces on the SIGN POST (Extreme I, gamma_W = 1.0) --------------------
# fabric tributary width = post spacing; the sign shadows its own fabric area
A_fabric_net = (s_post * H_fabric - A_sign) * solidity        # ft^2 of solid fabric
A_post       = d_post_ft * H_fabric                           # projected post area

F_sign   = Pz_sign   * A_sign          # lb
F_fabric = Pz_fabric * A_fabric_net
F_post   = Pz_post   * A_post

V_base = F_sign + F_fabric + F_post                                    # lb
M_base = (F_sign * z_sign_cg + F_fabric * H_fabric/2 + F_post * H_fabric/2)  # lb-ft

print(f"F_sign   = {F_sign:6.0f} lb  @ {z_sign_cg:.1f} ft")
print(f"F_fabric = {F_fabric:6.0f} lb  @ {H_fabric/2:.1f} ft  (net area {A_fabric_net:.1f} ft^2)")
print(f"F_post   = {F_post:6.0f} lb  @ {H_fabric/2:.1f} ft")
print(f"\nsign post base shear  V = {V_base:5.0f} lb")
print(f"sign post base moment M = {M_base:5.0f} lb-ft = {M_base*12/1000:.1f} kip-in")

F_sign   =    549 lb  @ 1.5 ft
F_fabric =    490 lb  @ 4.0 ft  (net area 13.0 ft^2)
F_post   =     66 lb  @ 4.0 ft

sign post base shear  V =  1105 lb
sign post base moment M =  3048 lb-ft = 36.6 kip-in


## Step 2 — Sign-to-fence connection

The panel is clamped to the sign post with **two 3/8-in U-bolts** (4 shear
legs) plus tie clips to the fabric that we ignore structurally.  LTS 5.15 sends
bolted-connection design to AISC — single shear on the threaded legs,
$\phi R_n = \phi\,F_{nv}\,A_b$ with $F_{nv}$ = 27 ksi (A307, threads included).

In [4]:
n_legs   = 4
V_leg    = F_sign / n_legs                     # lb per shear leg
A_b      = math.pi * 0.375**2 / 4              # in^2
phiRn_leg = 0.75 * 27_000 * A_b                # lb
dc_ubolt = V_leg / phiRn_leg
print(f"shear/leg = {V_leg:.0f} lb  vs  phiRn = {phiRn_leg:.0f} lb   ->  D/C = {dc_ubolt:.2f}")
# bearing on the 0.203-in pipe wall is even less critical at this load level

shear/leg = 137 lb  vs  phiRn = 2237 lb   ->  D/C = 0.06


## Step 3 — Fence post flexure

Each post is a vertical cantilever off the barrier. Check the standard line  
post (fabric only) and the sign post with the panel load.

Per the project specifications for Vandal Fencing, the fence line posts are  
**2.880-inch OD Grade 2 Pipe** with a yield strength of **50,000 psi**. 

Flexural resistance for compact round tubes: $\phi M_n = 0.9 M_p = 0.9 F_y Z$  
(LTS Article 5.8.2, Table 5.8.2-1). Using the plastic section modulus ($Z$)  
instead of the elastic section modulus ($S$) reclaims roughly 27% of the  
section's nominal capacity.

In [12]:
from civilpy.structural.steel import SteelSection

# --- site / spec inputs updated -----------------------------------------
Fy = 50_000.0   # psi (Updated per ODOT spec: Grade 2 pipe, Fy = 50,000 psi)

# --- line post (no sign): full fabric tributary, no shadowing ------------
F_fab_line = Pz_fabric * (s_post * H_fabric) * solidity
F_post_line = Pz_post * A_post
M_line = (F_fab_line + F_post_line) * H_fabric / 2 * 12     # lb-in

line_post = SteelSection("Pipe2-1/2SCH40")

# Switch to Plastic Section Modulus (Z) per LTS Table 5.8.2-1
Z_line = line_post.Z_x.magnitude                            # in^3
dc_line = M_line / (0.9 * Fy * Z_line)
print(f"line post  Pipe2-1/2SCH40: M = {M_line/1000:5.1f} kip-in, phiMn = {0.9*Fy*Z_line/1000:5.1f} kip-in -> D/C = {dc_line:.2f}")

# --- sign post: try the same section -------------------------------------
M_sign_post = M_base * 12                                   # lb-in
dc_sign_25 = M_sign_post / (0.9 * Fy * Z_line)

if dc_sign_25 <= 1.0:
    print(f"sign post  Pipe2-1/2SCH40: M = {M_sign_post/1000:5.1f} kip-in, phiMn = {0.9*Fy*Z_line/1000:5.1f} kip-in -> D/C = {dc_sign_25:.2f}  <-- PASSES!")
else:
    print(f"sign post  Pipe2-1/2SCH40: M = {M_sign_post/1000:5.1f} kip-in, phiMn = {0.9*Fy*Z_line/1000:5.1f} kip-in -> D/C = {dc_sign_25:.2f}  <-- FAILS")
    
    # Optional upsize if still failing
    sign_post = SteelSection("Pipe4SCH40")
    Z_sign = sign_post.Z_x.magnitude
    dc_sign = M_sign_post / (0.9 * Fy * Z_sign)
    print(f"sign post  Pipe4SCH40:     M = {M_sign_post/1000:5.1f} kip-in, phiMn = {0.9*Fy*Z_sign/1000:5.1f} kip-in -> D/C = {dc_sign:.2f}")

line post  Pipe2-1/2SCH40: M =  32.1 kip-in, phiMn =  61.7 kip-in -> D/C = 0.52
sign post  Pipe2-1/2SCH40: M =  36.6 kip-in, phiMn =  61.7 kip-in -> D/C = 0.59  <-- PASSES!


A quick service check (Service I uses the 10-yr wind, Fig. 3.8-4):

In [13]:
I_sign = sign_post.I_x.magnitude        # in^4
E = 29_000_000.0
scale10 = (V10 / V)**2 / 1.0            # pressures scale with V^2 (Cv regime unchanged)
# treat the total as a tip load at the resultant height for a quick bound
z_res = M_base / V_base                  # ft
P_serv = V_base * scale10
delta = P_serv * (z_res*12)**3 / (3 * E * I_sign)
print(f"10-yr wind resultant {P_serv:.0f} lb at {z_res:.1f} ft -> tip deflection ~ {delta:.2f} in "
      f"({z_res*12/delta:.0f} = L/{z_res*12/delta:.0f})")

10-yr wind resultant 483 lb at 2.8 ft -> tip deflection ~ 0.03 in (1123 = L/1123)


## Step 4 — Fabric-to-post connection

The fabric delivers its share of wind to each post through **tension bands**  
(1" × 1/8-in bands with 3/8-in A307 carriage bolts).  With bands at 12 in  
o.c. (8 per post) plus top/bottom tie wires, each band bolt sees the fabric  
reaction over its tributary height, in single shear.

\# //TODO - Not sure about this, the drawing shows 6/7 bands per post but says 12" max

In [15]:
n_bands = 6
V_band = F_fabric / n_bands                    # lb per band bolt (sign post, worst)
A_b38 = math.pi * 0.375**2 / 4                 # in^2
phiRn_band = 0.75 * 27_000 * A_b38
dc_band = V_band / phiRn_band
print(f"band bolt shear = {V_band:.0f} lb vs phiRn = {phiRn_band:.0f} lb -> D/C = {dc_band:.2f}")
print("(the 11-ga fabric selvage and hog rings are good for far more than "
      f"{F_fabric/ (s_post*H_fabric):.1f} psf average — not the weak link)")

band bolt shear = 82 lb vs phiRn = 2237 lb -> D/C = 0.04
(the 9-ga fabric selvage and hog rings are good for far more than 6.1 psf average — not the weak link)


## Step 5 — Base plate and anchor bolts

The sign post lands on an 8×8×3/4-in base plate with **four 3/4-in ASTM F1554
Gr 36 anchor bolts on a 5-in square pattern**, embedded 6 in into the 12-in-wide
barrier cap.  The base moment resolves into a bolt-row couple; the leeward pair
takes the tension.

LTS 5.16.3: *"the anchorage system shall be proportioned such that the load in
the steel portion of the anchorage will reach its minimum tensile strength
prior to failure of the concrete"* — design per AISC Design Guide 1 and the
ACI 318 anchoring provisions, which is exactly what
`civilpy.structural.concrete.AnchorBolts` implements (ACI 318-19 Ch. 17).

\# //TODO - These assumptions were very wrong. Here's is the actual specs,

There are three "standard" base plates at ODOT, an 8.5" x13" plate with 4  
holes, but the railing is offset from the centerline. A 10"x13" base plate  
also with the anchors and post offset from the centerline, and a wrap  
around plate where the plate is bent around the top corner of the parapet,  
with two vertical anchor bolts, and two horizontal anchor bolts.

Assuming the 8.5"x13" plate controls for the 10" plate as well, 

all anchor bolts have a 7" minimum embedment depth.

The anchor bolts are offset from the left edge (looking down) 3", then 4" from that one, leaving 1 1/2" to the right edge of the plate.
From the bottom of the plate, they're 1 1/2" up, 10" up from that, leaving 1 1/2" to the top edge.

The 2" diameter hole and post are centered vertically, but approximately 2 1/2" from the left edge of the plate.

For the wrap around base plate;

The vertical anchor is embedded 6" min, the horizontal is 4" 

There are again 8", 10", and now also 11" plates.

The 11" plate has 2 horizontal anchor bolts embedded 5 3/4", the other 2 have 1.

for the 8" plate the vertical anchors are offset 2 3/4" from the right edge, and 1 1/2" from the top and botttom, with 1' between them,
the same is true for the 10" anchor plate.

the horizontal bolt is centered on the plate, 8.5" down from the top of the parapet for both 8" and 10" configurations, 8" down for the 11". 

The post is again offset from the edge of the plate by approx 2.5" for the 11" plate, centered for the 8" plate, and 4" from the front face for the 10" plate. The drawings have been uploaded to snbi_ui/docs/notebooks/ODOT Ref for your reference as far as the specs are concerned. It's assumed you won't be able to pull dimensions.

  ADHESIVE ANCHORS SHALL NOT BE SUBSTITUTED WITH

AND UNCRACKED CONCRETE APPLICATIONS.

ACI 318 CHAPTER 17, "ANCHORING TO CONCRETE", FOR CRACKED

THE ANCHOR ADHESIVE SHALL BE EVALUATED ACCORDING TO

OR EXCEED THE CAPACITY OF THE ANCHORS LISTED ABOVE.

EVALUATED BY ICC-ES. THE SUBSTITUTE ANCHORS SHALL MEET

THE CONTRACTOR MAY USE A SUBSTITUTE ADHESIVE ANCHOR

(ICC-ES REPORT ESR-4094)

ATC ULTRABOND HS-1CC ADHESIVE ANCHOR SYSTEM

(ICC-ES REPORT ESR-4057)

SIMPSON STRONG-TIE SET-3G EPOXY ADHESIVE ANCHORS

(ICC-ES REPORT ESR-3298)

DEWALT PURE110+ EPOXY ADHESIVE ANCHOR SYSTEM

(ICC-ES REPORT ESR-3187)

HILTI HIT-HY 200 ADHESIVE ANCHORS

SELECT FROM ONE OF THE FOLLOWING APPROVED PRODUCTS:

WWW.ICC-ES.ORG/EVALUATION REPORTS/INDEX.SHTML

MANUFACTURER'S REQUIREMENTS FOR DRY CONCRETE.

CLEANED AND DRIED IN A MANNER CONSISTENT WITH THE

INSTALLATION OF THE ANCHORS, THE HOLES SHALL BE

WITH A HAMMER DRILL AND CARBIDE BIT. PRIOR TO THE

THE HOLES FOR THE ADHESIVE ANCHORS SHALL BE DRILLED

REPORTS LISTED BELOW.

ER'S INSTALLATION INSTRUCTIONS PUBLISHED IN THE ICC-ES

INSTALL ADHESIVE ANCHORS ACCORDING TO THE MANUFACTURANCHOR HARDWARE ACCODING TO ASTM B695, CLASS 65.

ASTM F436 WASHERS. MECHANICALLY GALVANIZE ALL

ASTM F1554, GRADE 36, WITH ASTM A563 NUTS AND

THE THREADED ROD FOR ADHESIVE ANCHORS SHALL B

In [8]:
s_bolt = 5.0                                   # in, both directions
T_row  = M_sign_post / s_bolt                  # lb, couple between bolt rows
N_ua   = T_row                                  # total tension on the 2-bolt row
V_ua   = V_base                                 # lb, whole group shear
print(f"bolt-row tension couple: T = M/s = {M_sign_post/1000:.1f} kip-in / {s_bolt:.0f} in = {T_row:.0f} lb "
      f"({T_row/2:.0f} lb/bolt)")
print(f"group shear V = {V_ua:.0f} lb ({V_ua/4:.0f} lb/bolt)")

bolt-row tension couple: T = M/s = 36.6 kip-in / 5 in = 7315 lb (3657 lb/bolt)
group shear V = 1105 lb (276 lb/bolt)


In [9]:
from civilpy.structural.concrete import AnchorBolts

# 12-in barrier top: bolts 5 in apart across the width -> 3.5 in to each face
anchors = AnchorBolts(
    f_c=4000.0, h_a=12.0,
    d_a=0.75, h_ef=6.0,
    f_ya=55_000.0, f_uta=75_000.0,          # F1554 Gr 55 (LTS Table 5.16.2-1)
    n_x=2, n_y=2, s_x=s_bolt, s_y=s_bolt,
    c_a1=3.5,                                # traffic-face edge, in shear direction
    c_a2=100.0,                              # barrier runs long parallel to fence
    A_brg=0.65,                              # heavy-hex head bearing area, in^2
    e_N_prime=s_bolt / 2,                    # tension resultant at the leeward row
    N_ua=N_ua, V_ua=V_ua,
    shear_direction="perpendicular",
    is_cracked=True, has_supp_reinf=False,
)
print(anchors.summary())

-----------------------------------------------------------------------------------------------------
Limit State                         Ref                           φSn (kip)     Demand     DCR Status
-----------------------------------------------------------------------------------------------------
Steel strength (tension)            ACI 318-19 Eq. 17.6.1.2           75.25       7.31   0.097     OK
Concrete breakout (tension)         ACI 318-19 Eq. 17.6.2.1b           4.44       7.31   1.649     NG
Pullout (tension)                   ACI 318-19 Eq. 17.6.3.2.2a (headed)      58.24       7.31   0.126     OK
Steel strength (shear)              ACI 318-19 Eq. 17.7.1.2b          39.13       1.11   0.028     OK
Concrete breakout (shear)           ACI 318-19 Eq. 17.7.2.1b           3.85       1.11   0.287     OK
Pryout (shear)                      ACI 318-19 Eq. 17.7.3.1            8.87       1.11   0.125     OK
Tension-shear interaction           ACI 318-19 Eq. 17.8.3             ≤1.20

**Second (governing) weak spot: concrete tension breakout.**  On a 12-in
barrier top the breakout cone has nowhere to go — 3.5-in edge distances gut the
projected area, and the check fails by a wide margin even though the *bolts
themselves* are barely working.  This also violates the LTS 5.16.3 ductility
requirement (concrete would fail long before the F1554 rods yield).

The standard fix is **anchor reinforcement** (ACI 318-19 17.5.2.1): hairpin
U-bars enclosing the anchors and lapped with the barrier verticals, sized to
take the full breakout load, so the concrete-breakout limit state is replaced
by developed rebar strength:

In [10]:
# 2x #4 hairpins per bolt row: 4 legs x 0.20 in^2 x 60 ksi, phi = 0.75
phiNn_hairpin = 0.75 * 4 * 0.20 * 60_000
anchors_fixed = AnchorBolts(
    f_c=4000.0, h_a=12.0, d_a=0.75, h_ef=6.0,
    f_ya=55_000.0, f_uta=75_000.0,
    n_x=2, n_y=2, s_x=s_bolt, s_y=s_bolt,
    c_a1=3.5, c_a2=100.0, A_brg=0.65, e_N_prime=s_bolt/2,
    N_ua=N_ua, V_ua=V_ua, shear_direction="perpendicular",
    is_cracked=True,
    anchor_reinf_tension=phiNn_hairpin,      # ACI 17.5.2.1 — replaces breakout
    anchor_reinf_shear=phiNn_hairpin,
)
print(anchors_fixed.summary())

-----------------------------------------------------------------------------------------------------
Limit State                         Ref                           φSn (kip)     Demand     DCR Status
-----------------------------------------------------------------------------------------------------
Steel strength (tension)            ACI 318-19 Eq. 17.6.1.2           75.25       7.31   0.097     OK
Anchor reinforcement (tension)      ACI 318-19 Section 17.5.2.1       36.00       7.31   0.203     OK
Pullout (tension)                   ACI 318-19 Eq. 17.6.3.2.2a (headed)      58.24       7.31   0.126     OK
Steel strength (shear)              ACI 318-19 Eq. 17.7.1.2b          39.13       1.11   0.028     OK
Anchor reinforcement (shear)        ACI 318-19 Section 17.5.2.1       36.00       1.11   0.031     OK
Pryout (shear)                      ACI 318-19 Eq. 17.7.3.1            8.87       1.11   0.125     OK
Tension-shear interaction           ACI 318-19 Eq. 17.8.3             ≤1.20

## Load-path summary

Every link, worst factored demand over design strength:

In [11]:
import pandas as pd

res = anchors_fixed.check_all()
rows = [
    ("1. wind pressure (LTS 3.8)",        f"{Pz_sign:.1f} psf on sign", "-", "-"),
    ("2. sign U-bolt clamps",             f"{V_leg:.0f} lb/leg",  f"{phiRn_leg:.0f} lb", dc_ubolt),
    ("3a. line post (fabric only)",       f"{M_line/1000:.1f} k-in", f"{0.9*Fy*S_line/1000:.1f} k-in", dc_line),
    ("3b. sign post Pipe2-1/2 (as-std)",  f"{M_sign_post/1000:.1f} k-in", f"{0.9*Fy*S_line/1000:.1f} k-in", dc_sign_25),
    ("3c. sign post Pipe4 (upsized)",     f"{M_sign_post/1000:.1f} k-in", f"{0.9*Fy*S_sign/1000:.1f} k-in", dc_sign),
    ("4. fabric tension-band bolts",      f"{V_band:.0f} lb", f"{phiRn_band:.0f} lb", dc_band),
    ("5a. anchors — no anchor reinf",     f"{N_ua:.0f} lb", "breakout governs", anchors.dcr()),
    ("5b. anchors — with hairpins",       f"{N_ua:.0f} lb", "steel/reinf governs", anchors_fixed.dcr()),
]
df = pd.DataFrame(rows, columns=["load-path link", "demand", "capacity", "D/C"])
df["status"] = df["D/C"].apply(lambda x: "-" if x == "-" else ("OK" if x <= 1.0 else "NG"))
df

AttributeError: 'AnchorBolts' object has no attribute 'dcr'

**Conclusions**

- The wind load itself is modest (~37 psf on the sign at 115 mph), but the
  3×5 solid sign concentrates ~550 lb near the top of an 8-ft lever arm.
- The **standard line post cannot carry the sign** — the post the sign hangs on
  must be upsized (or the sign moved off the fence).
- The **anchorage into the narrow barrier top is the governing weak spot**:
  concrete tension breakout fails outright and is non-ductile.  Anchor
  reinforcement (hairpins tied into the barrier cage), through-bolting, or
  embedding the posts into a widened cap are the realistic fixes; more/larger
  bolts do **not** help because concrete, not steel, governs.
- Everything transfers to the barrier as an added overturning line load —
  check the barrier/retaining wall and the deck overhang for the combined
  fence+sign moment per AASHTO LRFD BDS Art. 3.8.1.2.4 (sound-barrier analogy:
  resultant at 0.55·H) and Section 13/A13.4; attachments on a crash-tested
  railing also raise a MASH crashworthiness question for the Owner.

**Fatigue note (LTS Section 11):** attachments on flexible supports over
traffic should also see the Fatigue I natural-wind-gust check,
$P_{NW} = 5.2\,C_d\,I_F$ psf (Eq. 11.7.1.2-1):

In [ ]:
P_nw = 5.2 * Cd_sign          # psf, IF = 1.0 (typical)
M_nw = P_nw * A_sign * z_sign_cg * 12          # lb-in, sign only (dominant)
sr = M_nw / S_sign / 1000                       # ksi stress range at post base
print(f"P_NW = {P_nw:.1f} psf -> post-base stress range ~ {sr:.2f} ksi")
print("vs. CAFT: fillet-welded tube-to-base-plate ~ 4.5 ksi (Cat E'), anchor bolts 7 ksi (Cat D)")
print("-> fatigue OK by inspection here, but it governs quickly for larger signs/taller fences")